## Import packages

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
import sys

# Add the path where your tools.py is located
sys.path.append('/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset')

from tools import forward_fill_pipeline, normalize_dataframe, normalize_df_with_statistics


data_dir = "/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset"
Path(os.path.join(data_dir, 'processed')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(data_dir, 'statistics')).mkdir(parents=True, exist_ok=True)

SEED = 42

## Read data from files

### Record feature names

In [9]:
basic_records = ['PatientID', 'RecordTime', 'AdmissionTime', 'DischargeTime']
target_features = ['Outcome', 'LOS', 'Readmission']
demographic_features = ['Sex', 'Age'] # Sex and ICUType are binary features, others are continuous features
labtest_features = ['Capillary refill rate->0.0', 'Capillary refill rate->1.0',
        'Glascow coma scale eye opening->To Pain',
        'Glascow coma scale eye opening->3 To speech',
        'Glascow coma scale eye opening->1 No Response',
        'Glascow coma scale eye opening->4 Spontaneously',
        'Glascow coma scale eye opening->None',
        'Glascow coma scale eye opening->To Speech',
        'Glascow coma scale eye opening->Spontaneously',
        'Glascow coma scale eye opening->2 To pain',
        'Glascow coma scale motor response->1 No Response',
        'Glascow coma scale motor response->3 Abnorm flexion',
        'Glascow coma scale motor response->Abnormal extension',
        'Glascow coma scale motor response->No response',
        'Glascow coma scale motor response->4 Flex-withdraws',
        'Glascow coma scale motor response->Localizes Pain',
        'Glascow coma scale motor response->Flex-withdraws',
        'Glascow coma scale motor response->Obeys Commands',
        'Glascow coma scale motor response->Abnormal Flexion',
        'Glascow coma scale motor response->6 Obeys Commands',
        'Glascow coma scale motor response->5 Localizes Pain',
        'Glascow coma scale motor response->2 Abnorm extensn',
        'Glascow coma scale total->11', 'Glascow coma scale total->10',
        'Glascow coma scale total->13', 'Glascow coma scale total->12',
        'Glascow coma scale total->15', 'Glascow coma scale total->14',
        'Glascow coma scale total->3', 'Glascow coma scale total->5',
        'Glascow coma scale total->4', 'Glascow coma scale total->7',
        'Glascow coma scale total->6', 'Glascow coma scale total->9',
        'Glascow coma scale total->8',
        'Glascow coma scale verbal response->1 No Response',
        'Glascow coma scale verbal response->No Response',
        'Glascow coma scale verbal response->Confused',
        'Glascow coma scale verbal response->Inappropriate Words',
        'Glascow coma scale verbal response->Oriented',
        'Glascow coma scale verbal response->No Response-ETT',
        'Glascow coma scale verbal response->5 Oriented',
        'Glascow coma scale verbal response->Incomprehensible sounds',
        'Glascow coma scale verbal response->1.0 ET/Trach',
        'Glascow coma scale verbal response->4 Confused',
        'Glascow coma scale verbal response->2 Incomp sounds',
        'Glascow coma scale verbal response->3 Inapprop words',
        'Diastolic blood pressure', 'Fraction inspired oxygen', 'Glucose',
        'Heart Rate', 'Height', 'Mean blood pressure', 'Oxygen saturation',
        'Respiratory rate', 'Systolic blood pressure', 'Temperature', 'Weight',
        'pH']
require_impute_features = labtest_features
normalize_features = ['Age'] + ['Diastolic blood pressure', 'Fraction inspired oxygen', 'Glucose',
        'Heart Rate', 'Height', 'Mean blood pressure', 'Oxygen saturation',
        'Respiratory rate', 'Systolic blood pressure', 'Temperature', 'Weight',
        'pH'] + ['LOS']

In [10]:
df = pd.read_csv(os.path.join(data_dir, "processed", f"format_mimic4_ehr.csv"))
# df

In [11]:
# if a patient has multiple records, we only use the first 48 items
# we also discard the patients with less than 48 items

# Ensure dataframe is sorted by PatientID and RecordTime
df = df.sort_values(['PatientID', 'RecordTime'])

# Filter out patients with less than 48 records
df = df.groupby('PatientID').filter(lambda x: len(x) >= 48)

# Select the first 48 records for each patient
df = df.groupby('PatientID').head(48)

df


,PatientID,RecordTime,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Decompensation,Acute and unspecified renal failure,Acute cerebrovascular disease,...,Glucose,Heart Rate,Height,Mean blood pressure,Oxygen saturation,Respiratory rate,Systolic blood pressure,Temperature,Weight,pH
101,10001884_1,1,2131-01-11 04:20:05,2131-01-20 08:27:30,1.0,219.123611,1.0,0.0,0.0,0.0,...,NaN,60.0,NaN,70.0,98.0,10.0,167.0,NaN,NaN,NaN
102,10001884_1,2,2131-01-11 04:20:05,2131-01-20 08:27:30,1.0,218.123611,1.0,0.0,0.0,0.0,...,NaN,72.0,NaN,75.0,100.0,20.0,102.0,NaN,NaN,NaN
103,10001884_1,3,2131-01-11 04:20:05,2131-01-20 08:27:30,1.0,217.123611,1.0,0.0,0.0,0.0,...,140.0,70.0,NaN,73.0,100.0,20.0,93.0,35.4,NaN,7.33
104,10001884_1,4,2131-01-11 04:20:05,2131-01-20 08:27:30,1.0,216.123611,1.0,0.0,0.0,0.0,...,NaN,71.0,NaN,80.0,100.0,20.0,102.0,35.8,NaN,NaN
105,10001884_1,5,2131-01-11 04:20:05,2131-01-20 08:27:30,1.0,215.123611,1.0,0.0,0.0,0.0,...,NaN,71.0,NaN,92.0,98.0,20.0,138.0,36.3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1557837,13886221_1,44,2152-08-02 04:57:56,2152-08-04 17:55:11,0.0,16.954167,0.0,0.0,0.0,0.0,...,NaN,91.0,NaN,111.0,99.0,18.0,144.0,36.5,NaN,NaN
1557838,13886221_1,45,2152-08-02 04:57:56,2152-08-04 17:55:11,0.0,15.954167,0.0,0.0,0.0,0.0,...,NaN,101.0,NaN,79.0,95.0,5.0,111.0,NaN,NaN,NaN
1557839,13886221_1,46,2152-08-02 04:57:56,2152-08-04 17:55:11,0.0,14.954167,0.0,0.0,0.0,0.0,...,126.0,95.0,NaN,96.0,97.0,8.0,123.0,NaN,NaN,NaN
1557840,13886221_1,47,2152-08-02 04:57:56,2152-08-04 17:55:11,0.0,13.954167,0.0,0.0,0.0,0.0,...,NaN,99.0,NaN,101.0,98.0,21.0,134.0,NaN,NaN,NaN


In [12]:
def assign_group(rt):
    return (rt - 1) // 12

# Assign group for each record
df['Group'] = df['RecordTime'].apply(assign_group)

aggregation_logic = {}
for f in demographic_features+labtest_features:
    aggregation_logic[f] = 'mean'

aggregation_logic['Outcome'] = 'mean'
aggregation_logic['LOS'] = 'mean'
aggregation_logic['Readmission'] = 'mean'


# Group by PatientID and Group, then aggregate
aggregated_df = df.groupby(['PatientID', 'Group']).agg(aggregation_logic)  # replace 'mean' with your desired aggregation

# Reset index if needed
aggregated_df = aggregated_df.reset_index()

aggregated_df

,PatientID,Group,Sex,Age,Capillary refill rate->0.0,Capillary refill rate->1.0,Glascow coma scale eye opening->To Pain,Glascow coma scale eye opening->3 To speech,Glascow coma scale eye opening->1 No Response,Glascow coma scale eye opening->4 Spontaneously,...,Mean blood pressure,Oxygen saturation,Respiratory rate,Systolic blood pressure,Temperature,Weight,pH,Outcome,LOS,Readmission
0,10001884_1,0,0.0,68.0,NaN,NaN,0.2,0.0,0.0,0.0,...,84.083333,98.727273,17.916667,115.000000,36.411111,NaN,7.340000,1.0,213.623611,1.0
1,10001884_1,1,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,91.500000,99.000000,18.583333,128.916667,36.720000,NaN,7.340000,1.0,201.623611,1.0
2,10001884_1,2,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,94.250000,98.272727,20.250000,131.833333,36.908333,NaN,NaN,1.0,189.623611,1.0
3,10001884_1,3,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,103.250000,94.416667,23.166667,153.083333,36.761111,NaN,7.400000,1.0,177.623611,1.0
4,10002155_1,-1,0.0,80.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.000000,NaN,NaN,53.0,NaN,0.0,148.293889,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39630,13886208_1,3,0.0,39.0,NaN,NaN,0.0,0.0,0.0,0.0,...,77.800000,98.833333,22.727273,114.100000,37.481481,NaN,NaN,0.0,16.181667,0.0
39631,13886221_1,0,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,77.454545,99.818182,20.727273,100.363636,37.444444,54.0,7.250000,0.0,54.454167,0.0
39632,13886221_1,1,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,75.750000,100.000000,23.000000,102.166667,36.472222,NaN,7.356667,0.0,42.454167,0.0
39633,13886221_1,2,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,86.000000,99.333333,17.666667,116.750000,36.444444,58.2,7.405000,0.0,30.454167,0.0


In [13]:
aggregated_df = aggregated_df.rename(columns={'Group':'RecordTime'})
df = aggregated_df

df['LOS'] = 5-df['RecordTime']
df

,PatientID,RecordTime,Sex,Age,Capillary refill rate->0.0,Capillary refill rate->1.0,Glascow coma scale eye opening->To Pain,Glascow coma scale eye opening->3 To speech,Glascow coma scale eye opening->1 No Response,Glascow coma scale eye opening->4 Spontaneously,...,Mean blood pressure,Oxygen saturation,Respiratory rate,Systolic blood pressure,Temperature,Weight,pH,Outcome,LOS,Readmission
0,10001884_1,0,0.0,68.0,NaN,NaN,0.2,0.0,0.0,0.0,...,84.083333,98.727273,17.916667,115.000000,36.411111,NaN,7.340000,1.0,5,1.0
1,10001884_1,1,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,91.500000,99.000000,18.583333,128.916667,36.720000,NaN,7.340000,1.0,4,1.0
2,10001884_1,2,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,94.250000,98.272727,20.250000,131.833333,36.908333,NaN,NaN,1.0,3,1.0
3,10001884_1,3,0.0,68.0,NaN,NaN,0.0,0.0,0.0,0.0,...,103.250000,94.416667,23.166667,153.083333,36.761111,NaN,7.400000,1.0,2,1.0
4,10002155_1,-1,0.0,80.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.000000,NaN,NaN,53.0,NaN,0.0,6,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39630,13886208_1,3,0.0,39.0,NaN,NaN,0.0,0.0,0.0,0.0,...,77.800000,98.833333,22.727273,114.100000,37.481481,NaN,NaN,0.0,2,0.0
39631,13886221_1,0,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,77.454545,99.818182,20.727273,100.363636,37.444444,54.0,7.250000,0.0,5,0.0
39632,13886221_1,1,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,75.750000,100.000000,23.000000,102.166667,36.472222,NaN,7.356667,0.0,4,0.0
39633,13886221_1,2,0.0,34.0,NaN,NaN,0.0,0.0,0.0,0.0,...,86.000000,99.333333,17.666667,116.750000,36.444444,58.2,7.405000,0.0,3,0.0


## Stratified split dataset into `Training`, `Validation` and `Test` sets (ML/DL settings)

- Stratified dataset according to `Outcome` column
- 50% Training, 45% Validation, 5% Test
  - Name: train, val, test


In [14]:
# Group the dataframe by patient ID
grouped = df.groupby('PatientID')

# Get the patient IDs and outcomes
patients = np.array(list(grouped.groups.keys()))
patients_outcome = np.array([grouped.get_group(patient_id)['Outcome'].iloc[0] for patient_id in patients])

# Get the train_val/test patient IDs
train_val_patients, test_patients = train_test_split(patients, test_size=1/100, random_state=SEED, stratify=patients_outcome)

# Get the train/val patient IDs
train_val_patients_outcome = np.array([grouped.get_group(patient_id)['Outcome'].iloc[0] for patient_id in train_val_patients])
train_patients, val_patients = train_test_split(train_val_patients, test_size=49/99, random_state=SEED, stratify=train_val_patients_outcome)
# Create train, val, test dataframes for the current fold
train_df = df[df['PatientID'].isin(train_patients)]
val_df = df[df['PatientID'].isin(val_patients)]
test_df = df[df['PatientID'].isin(test_patients)]
save_dir = os.path.join(data_dir, 'processed', 'fold_ml') # forward fill
Path(save_dir).mkdir(parents=True, exist_ok=True)

# # Save the train, val, and test dataframes for the current fold to csv files
# train_df.to_csv(os.path.join(save_dir, "train_raw.csv"), index=False)
# val_df.to_csv(os.path.join(save_dir, "val_raw.csv"), index=False)
# test_df.to_csv(os.path.join(save_dir, "test_raw.csv"), index=False)
# Calculate the mean and std of the train set (include age, lab test features, and LOS) on the data in 5% to 95% quantile range
train_df, val_df, test_df, default_fill, los_info, train_mean, train_std = normalize_dataframe(train_df, val_df, test_df, normalize_features)

# # Save the zscored dataframes to csv files
# train_df.to_csv(os.path.join(save_dir, "train_after_zscore.csv"), index=False)
# val_df.to_csv(os.path.join(save_dir, "val_after_zscore.csv"), index=False)
# test_df.to_csv(os.path.join(save_dir, "test_after_zscore.csv"), index=False)

# Forward Imputation after grouped by PatientID
# Notice: if a patient has never done certain lab test, the imputed value will be the median value calculated from train set
train_x, train_y, train_pid, _, _ = forward_fill_pipeline(train_df, default_fill, demographic_features, labtest_features, target_features, require_impute_features)
val_x, val_y, val_pid, _, _ = forward_fill_pipeline(val_df, default_fill, demographic_features, labtest_features, target_features, require_impute_features)
test_x, test_y, test_pid, _, _ = forward_fill_pipeline(test_df, default_fill, demographic_features, labtest_features, target_features, require_impute_features)

# Save the imputed dataset to pickle file
pd.to_pickle(train_x, os.path.join(save_dir, "train_x.pkl"))
pd.to_pickle(train_y, os.path.join(save_dir, "train_y.pkl"))
pd.to_pickle(train_pid, os.path.join(save_dir, "train_pid.pkl"))
pd.to_pickle(val_x, os.path.join(save_dir, "val_x.pkl"))
pd.to_pickle(val_y, os.path.join(save_dir, "val_y.pkl"))
pd.to_pickle(val_pid, os.path.join(save_dir, "val_pid.pkl"))
pd.to_pickle(test_x, os.path.join(save_dir, "test_x.pkl"))
pd.to_pickle(test_y, os.path.join(save_dir, "test_y.pkl"))
pd.to_pickle(test_pid, os.path.join(save_dir, "test_pid.pkl"))
pd.to_pickle(los_info, os.path.join(save_dir, "los_info.pkl")) # LOS statistics (calculated from the train set)

/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset/tools.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.99984283  0.99994761 -0.99994761 ...  0.99994761 -0.99994761
 -2.99984283]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, normalize_features] = (train_df.loc[:, normalize_features] - train_mean) / (train_std+1e-12)
/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset/tools.py:137: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.99984283  0.99994761 -0.99994761 ...  0.99994761 -0.99994761
 -2.99984283]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, normalize_features] = (val_df.loc[:, normalize_features] - train_mean) / (train_std+1e-12)
/content/drive/M

### Hold-out dataset setting (Stratified) LLM settings

- 50% training, 45% validation, 5% testing


In [19]:
# Group the dataframe by patient ID
grouped = df.groupby('PatientID')

# Get the patient IDs and outcomes
patients = np.array(list(grouped.groups.keys()))
patients_outcome = np.array([grouped.get_group(patient_id)['Outcome'].iloc[0] for patient_id in patients])

# Get the train_val/test patient IDs
train_val_patients, test_patients = train_test_split(patients, test_size=1/100, random_state=SEED, stratify=patients_outcome)

# Get the train/val patient IDs
train_val_patients_outcome = np.array([grouped.get_group(patient_id)['Outcome'].iloc[0] for patient_id in train_val_patients])
train_patients, val_patients = train_test_split(train_val_patients, test_size=49/99, random_state=SEED, stratify=train_val_patients_outcome)
# Create train, val, test dataframes for the current fold
train_df = df[df['PatientID'].isin(train_patients)]
val_df = df[df['PatientID'].isin(val_patients)]
test_df = df[df['PatientID'].isin(test_patients)]
save_dir = os.path.join(data_dir, 'processed', 'fold_llm') # forward fill
Path(save_dir).mkdir(parents=True, exist_ok=True)


# Calculate the mean and std of the train set (include age, lab test features, and LOS) on the data in 5% to 95% quantile range
default_fill = normalize_dataframe(train_df, val_df, test_df, normalize_features, require_norm_later=False)

# Forward Imputation after grouped by PatientID
# Notice: if a patient has never done certain lab test, the imputed value will be the median value calculated from train set

test_x, test_y, test_pid, test_x_record_times, test_x_missing_masks = forward_fill_pipeline(test_df, default_fill, demographic_features, labtest_features, target_features, require_impute_features)

# Save the imputed dataset to pickle file
pd.to_pickle(test_x, os.path.join(save_dir, "test_x.pkl"))
pd.to_pickle(test_y, os.path.join(save_dir, "test_y.pkl"))
pd.to_pickle(test_pid, os.path.join(save_dir, "test_pid.pkl"))
pd.to_pickle(los_info, os.path.join(save_dir, "los_info.pkl")) # LOS statistics (calculated from the train set)
pd.to_pickle(test_x_record_times, os.path.join(save_dir, "test_x_record_times.pkl"))
pd.to_pickle(test_x_missing_masks, os.path.join(save_dir, "test_x_missing_masks.pkl"))

all_features = demographic_features + labtest_features
pd.to_pickle(all_features, os.path.join(save_dir, "all_features.pkl"))

In [22]:
test_patients = pd.read_pickle('/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset/processed/fold_llm/test_pid.pkl')
test_df = df[df['PatientID'].isin(test_patients)]
test_x, test_y, test_pid, test_x_record_times, test_x_missing_masks = forward_fill_pipeline(test_df, None, demographic_features, labtest_features, target_features, [])
pd.to_pickle(test_x_missing_masks, "/content/drive/MyDrive/master year 1 section 2/advance ML/project/workspace/Dataset/processed/fold_llm/test_x_missing_masks.pkl")